## MLflow's Model Registry

In [26]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)  # Adjust port/URL if running on a remote host

### Interacting with the MLflow tracking server

The `MlflowClient` object allows us to interact with...
- an MLflow Tracking Server that creates and manages experiments and runs.
- an MLflow Registry Server that creates and manages registered models and model versions. 

To instantiate it we need to pass a tracking URI and/or a registry URI

In [32]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)



In [28]:
runs = mlflow.search_runs(experiment_names=["nyc-taxi-experiment"])
print(runs[["run_id"]])

                              run_id
0   bd9383f22fab4c4a96e546f38244eef7
1   185631ca8e9f4612ab5e72d51ecbee2f
2   8c13c15ef6d1492cba0412ba1679c63e
3   0623dfa974434e6e94a4da650969f3c1
4   c46d96e2530b45a6ba053398da8eed2a
5   cc9d92685fae4814a8642ec621e8a965
6   923db63eab5843f294ed1e2d604fcc44
7   c59f039d61ad4e97a0d3ef1078b859d1
8   582af2d80205420fa2240247fed8ec31
9   80916b076d344f47ada451e9c3854133
10  04a74ae447cb4097994eed579df9b6ab
11  760c6f43584c4bc59184a4b1e65f158d
12  3a80673a8c3f439f8322a955fe9acd75
13  1ab7320f13494ec584f8753cb1efd2ab
14  85705457bedf4749b8cecde7ed97301d
15  ceae8644d3684638b4a1f2ce7bc2f01e
16  a3238f2fc1f344f59c8f8cdcff205f46
17  35720d2dc3f445779291d42c0778a888
18  2eaee1d3394a47efa4ef1d55585c0348
19  632d20e4ad9d467cbb5389506d5ad4f0
20  bac56fda5bfe43a1adcba9b557ec4440
21  2a7bf053b43841a3b4e7d5fd119e77fe
22  be342479058d4e1496fa7597064e8acc
23  2f886c4751d44d96b880d4a32e89faf2
24  b1ea1fa5b60849c3a6b3336245958579


Let's check the latest versions for the experiment with id `1`...

In [33]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids = 1,
     filter_string="metrics.rmse > 40",
        
        order_by=["metrics.rmse ASC"]
   
)

In [34]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 2a7bf053b43841a3b4e7d5fd119e77fe, rmse: 40.6227
run id: bac56fda5bfe43a1adcba9b557ec4440, rmse: 41.5494
run id: cc9d92685fae4814a8642ec621e8a965, rmse: 41.5935
run id: 923db63eab5843f294ed1e2d604fcc44, rmse: 41.6797
run id: 582af2d80205420fa2240247fed8ec31, rmse: 43.8290
run id: 80916b076d344f47ada451e9c3854133, rmse: 44.0207
run id: 04a74ae447cb4097994eed579df9b6ab, rmse: 44.0227
run id: a3238f2fc1f344f59c8f8cdcff205f46, rmse: 44.0439
run id: 3a80673a8c3f439f8322a955fe9acd75, rmse: 44.0554
run id: 35720d2dc3f445779291d42c0778a888, rmse: 44.6881
run id: ceae8644d3684638b4a1f2ce7bc2f01e, rmse: 45.1386
run id: 1ab7320f13494ec584f8753cb1efd2ab, rmse: 45.2138
run id: 85705457bedf4749b8cecde7ed97301d, rmse: 45.6013
run id: 760c6f43584c4bc59184a4b1e65f158d, rmse: 45.6232
run id: 2eaee1d3394a47efa4ef1d55585c0348, rmse: 147.4581
run id: be342479058d4e1496fa7597064e8acc, rmse: 147.4581
run id: 2f886c4751d44d96b880d4a32e89faf2, rmse: 147.4581


### Interacting with the Model Registry

In this section We will use the `MlflowClient` instance to:

1. Register a new version for the experiment `nyc-taxi-regressor`
2. Retrieve the latests versions of the model `nyc-taxi-regressor` and check that a new version `4` was created.
3. Transition the version `4` to "Staging" and adding annotations to it.

In [30]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(mlflow.get_active_model_id())

None


In [31]:
run_id = "cc9d92685fae4814a8642ec621e8a965"
model_uri = f"runs:/{run_id}/models_mlflow"
mlflow.register_model(model_uri=model_uri, name="registered-by-mlflowclient")

Successfully registered model 'registered-by-mlflowclient'.
2026/09/17 12:11:52 WARNING mlflow.tracking._model_registry.fluent: Run with id cc9d92685fae4814a8642ec621e8a965 has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-813b1fd08c2f47e29deba8d3320d7934 instead
2026/09/17 12:11:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: registered-by-mlflowclient, version 1
Created version '1' of model 'registered-by-mlflowclient'.


<ModelVersion: aliases=[], creation_timestamp=1789647112087, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1789647112087, metrics=None, model_id=None, name='registered-by-mlflowclient', params=None, run_id='cc9d92685fae4814a8642ec621e8a965', run_link='', source='models:/m-813b1fd08c2f47e29deba8d3320d7934', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [ ]:
model_name = "best_xgboost"
latest_versions = client.get_latest_versions(name=model_name)

client.set_registered_model_alias(
    name="best_xgboost", alias="staging", version="2"  # Or "production"
)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")
    

version: 2, stage: None


/tmp/ipykernel_3333/3751444238.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [ ]:
client.create_registered_model("xgboostbyclient")

In [43]:
run_id = "923db63eab5843f294ed1e2d604fcc44"
result = client.create_model_version(
    name="xgboostbyclient",
    source=f"runs:/{run_id}/models_mlflow",
    run_id=run_id,
)

2026/09/17 13:00:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboostbyclient, version 1


In [44]:
model_version = 1
new_stage = "Staging"
client.transition_model_version_stage(
    name="xgboostbyclient",
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_3333/4103549796.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1789650002508, current_stage='Staging', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1789650962790, metrics=None, model_id=None, name='xgboostbyclient', params=None, run_id='923db63eab5843f294ed1e2d604fcc44', run_link='', source='runs:/923db63eab5843f294ed1e2d604fcc44/models_mlflow', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [ ]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

### Comparing versions and selecting the new "Production" model

In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the `DictVectorizer` that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the `DictVectorizer` so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.


**Note: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.**

In [ ]:
from sklearn.metrics import mean_squared_error
import pandas as pd


def read_dataframe(filename):
    df = pd.read_csv(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": mean_squared_error(y_test, y_pred, squared=False)}

In [ ]:
df = read_dataframe("data/green_tripdata_2021-03.csv")

In [ ]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

In [ ]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [ ]:
X_test = preprocess(df, dv)

In [ ]:
target = "duration"
y_test = df[target].values

In [ ]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

In [ ]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=4,
    stage="Production",
    archive_existing_versions=True
)